In [43]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt
import json
from functions.plot_function import (plot_joint_trajectory,
                                     plot_joints_trajectory,
                                     plot_joint_index,
                                     animate_joint_index,)
from functions.load_data import load_data, load_multi_data
from functions.load_optitrack import (load_optitrack_csv, load_multi_optitrack,
                                      set_initial_position)
import csv
import pandas as pd

In [ ]:
BASE = os.path.join(os.getcwd(), 'logs')
FOLDER = 'cot3'
TRIALS = [0,1,2,3,4]

CASES = ['flat', 's20l', 's10lr']
MODELS = ['ff', 'hebb']

# raw servo logs and optitrack trajectories, zipped by index: raw[name][i]
# and runs[name][i] are always the same physical trial.
raw = {
    f'{case}_{model}': load_multi_data(case, FOLDER, model, TRIALS, BASE)
    for case in CASES for model in MODELS
}

runs = {
    f'{case}_{model}': set_initial_position(load_multi_optitrack(case, FOLDER, model, TRIALS, BASE))
    for case in CASES for model in MODELS
}

In [ ]:
GECKO_MASS = 2.45  # kg

RAW_TO_CURRENT = 2.69  # mA
TORQUE_CONSTANT = 0.5494505495
IDLE_CURRENT = 0.0

RAW_TO_RPM = 0.229
RPM_TO_RADPS = 2 * np.pi / 60

In [ ]:
def calculate_cot(trial, mass, distance):
    current = (np.array(trial['current']) * RAW_TO_CURRENT) / 1000.0  # A
    torque = (current - IDLE_CURRENT) / TORQUE_CONSTANT  # N*m

    radps = np.array(trial['vel']) * RAW_TO_RPM * RPM_TO_RADPS  # rad/s

    power = np.abs(torque * radps)  # W
    energy = np.sum(np.sum(power, axis=1) * np.array(trial['dt']))  # J

    return energy / (mass * 9.81 * distance)  # J/kg/m

In [ ]:
# Per-trial distance = farthest forward (x-only) reach: x_rel starts at 0 and
# goes more negative as the robot moves forward, so the farthest forward
# point is x_rel.min(), not raw pos_x (arbitrary start offset).
max_forward_distance = {name: np.array([-d['x_rel'].min() for d in takes])
                         for name, takes in runs.items()}

# Alternative: forward position at the final sample instead of the farthest
# point reached. Where the robot ends up can be less than its farthest
# reach if it drifts back before the log ends.
final_distance = {name: np.array([-d['x_rel'].iloc[-1] for d in takes])
                   for name, takes in runs.items()}

In [ ]:
cot = {name: np.array([calculate_cot(raw[name][i], GECKO_MASS, max_forward_distance[name][i])
                        for i in range(len(raw[name]))])
       for name in raw}

cot_final = {name: np.array([calculate_cot(raw[name][i], GECKO_MASS, final_distance[name][i])
                              for i in range(len(raw[name]))])
             for name in raw}

In [50]:
from scipy import stats
def mean_ci95(x):
    x = np.asarray(x)
    n = len(x)
    m = x.mean()
    sem = x.std(ddof=1) / np.sqrt(n)
    h = sem * stats.t.ppf(0.975, n - 1)  # 95% CI half-width, n=10 -> t~2.26
    return m, h

In [ ]:
INK_MUTED, GRID = '#52514e', '#e5e4e0'
BAR_COLORS = {'Feedforward': '#2a78d6', 'Hebbian': '#eb6834'}
DOT_COLORS = {'Feedforward': "#2d2ad6", 'Hebbian': "#e70e06"}

def style_ax(ax):
    ax.grid(True, axis='y', color=GRID, lw=0.8); ax.set_axisbelow(True)
    for s in ('top', 'right'): ax.spines[s].set_visible(False)
    for s in ('left', 'bottom'): ax.spines[s].set_color(GRID)
    ax.tick_params(colors=INK_MUTED)
    ax.xaxis.label.set_color(INK_MUTED); ax.yaxis.label.set_color(INK_MUTED)

def plot_distance_bars(groups, case, ylabel, title):
    n_trials = len(next(iter(groups.values()))[0])
    x = np.arange(len(case))
    width = 0.35

    fig, ax = plt.subplots(layout='constrained')
    for i, (label, arrs) in enumerate(groups.items()):
        means, halfwidths = zip(*(mean_ci95(a) for a in arrs))
        offset = (i - (len(groups) - 1) / 2) * width
        ax.bar(x + offset, means, width, label=label, color=BAR_COLORS[label])

        for xi, m, h in zip(x + offset, means, halfwidths):
            ax.text(xi, m + h, f'{m:.2f}', ha='center', va='bottom', fontsize=10)

        for xi, a in zip(x + offset, arrs):
            ax.scatter(np.full(len(a), xi), a, color=DOT_COLORS[label],
                       alpha=0.9, s=40, edgecolors='white', linewidth=1.2, zorder=3)

    ax.set_xticks(x, case)
    ax.set_ylabel(ylabel)
    ax.set_title(f'{title} (mean ± 95% CI, n={n_trials} trials)')
    style_ax(ax)
    # headroom so the value labels clear the legend
    top = max(m + h for arrs in groups.values() for m, h in (mean_ci95(a) for a in arrs))
    ax.set_ylim(0, top * 1.22)
    ax.legend(loc='upper left', ncols=2, frameon=False, labelcolor=INK_MUTED)

    plt.show()
    return fig, ax

In [ ]:
case = ("Flat",  "Rubber Left" , "Rubber Left Right")
groups = {
    'Feedforward': [cot["flat_ff"],   cot["s20l_ff"],   cot["s10lr_ff"]],
    'Hebbian':     [cot["flat_hebb"], cot["s20l_hebb"], cot["s10lr_hebb"]],
}

plot_distance_bars(groups, case, 'CoT (J/kg/m)', 'Cost of Transport')

In [ ]:
case = ("Flat",  "Rubber Left" , "Rubber Left Right")
groups = {
    'Feedforward': [cot_final["flat_ff"],   cot_final["s20l_ff"],   cot_final["s10lr_ff"]],
    'Hebbian':     [cot_final["flat_hebb"], cot_final["s20l_hebb"], cot_final["s10lr_hebb"]],
}

plot_distance_bars(groups, case, 'CoT (J/kg/m)', 'Cost of Transport, final-point distance')